# Research 03 — Balanced CatBoost

**Hypothesis:** balanced class weights may improve the recall/F1 tradeoff for the smaller Social and Whale segments. This is one predeclared loss-weighting experiment, not a search. Keep the CatBoost architecture and fixed three-fold seed identical to the existing OOF benchmark. Test is loaded only if the candidate probability blend improves OOF macro F1.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)
base = pd.read_csv(ROOT / "predictions/segment-oof.csv")
assert len(base) == len(train) and np.array_equal(base.segment.to_numpy(), y)
p_base = base[[f"prob_{i}" for i in range(4)]].to_numpy()
raw = train.drop(columns=["segment", "player_id"]).copy()
cat_cols = raw.select_dtypes(include=["object", "string"]).columns.tolist()
raw[cat_cols] = raw[cat_cols].fillna("__MISSING__").astype(str)
print(f"Rows={len(train):,}; class counts={np.bincount(y).tolist()}; categoricals={len(cat_cols)}")

Rows=104,000; class counts=[40934, 26034, 21014, 16018]; categoricals=10


## Three-fold OOF with native categorical handling

The failed parallel screen omitted `cat_features` and therefore did not produce evidence. This corrected run explicitly identifies categorical columns and keeps missing values as a stable category sentinel.

In [2]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
p_balanced = np.zeros((len(train), 4), dtype=float)
for fold, (fit_idx, valid_idx) in enumerate(cv.split(raw, y), start=1):
    model = CatBoostClassifier(
        iterations=300,
        depth=7,
        learning_rate=0.08,
        loss_function="MultiClass",
        auto_class_weights="Balanced",
        random_seed=42,
        thread_count=4,
        allow_writing_files=False,
        verbose=False,
    )
    model.fit(raw.iloc[fit_idx], y[fit_idx], cat_features=cat_cols)
    p_balanced[valid_idx] = model.predict_proba(raw.iloc[valid_idx])
    print(
        f"Fold {fold}: macro F1={f1_score(y[valid_idx], p_balanced[valid_idx].argmax(1), average='macro'):.5f}",
        flush=True,
    )

p_candidate = 0.8 * p_base + 0.2 * p_balanced


def score_row(name, probabilities):
    pred = probabilities.argmax(axis=1)
    return {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }


results = pd.DataFrame(
    [
        score_row("Existing CatBoost/LightGBM blend", p_base),
        score_row("Balanced CatBoost", p_balanced),
        score_row("80% existing + 20% balanced CatBoost", p_candidate),
    ]
)
display(results)
results.to_csv(ROOT / "assets/segment-balanced-catboost-results.csv", index=False)
pd.DataFrame(p_balanced, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-balanced-catboost-oof.csv", index=False
)
pd.DataFrame(p_candidate, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-balanced-catboost-blend-oof.csv", index=False
)
promote = (
    results.loc[results.candidate == "80% existing + 20% balanced CatBoost", "macro_f1"].iloc[0]
    > results.loc[results.candidate == "Existing CatBoost/LightGBM blend", "macro_f1"].iloc[0]
)
print("OOF promotion decision:", promote)

Fold 1: macro F1=0.74253


Fold 2: macro F1=0.73834


Fold 3: macro F1=0.74438


,candidate,macro_f1,weighted_f1
0,Existing CatBoost/LightGBM blend,0.750895,0.764683
1,Balanced CatBoost,0.741759,0.753181
2,80% existing + 20% balanced CatBoost,0.750872,0.764394


OOF promotion decision: False


## Prediction-only test pass when validated

If the fixed blend improves the saved OOF candidate, refit weighted CatBoost on all labeled rows, combine it with the existing all-data CatBoost/LightGBM checkpoint predictions, and save a keyed candidate file. No submission file or ID mapping is assumed.

In [3]:
if promote:
    final_balanced = CatBoostClassifier(
        iterations=300,
        depth=7,
        learning_rate=0.08,
        loss_function="MultiClass",
        auto_class_weights="Balanced",
        random_seed=42,
        thread_count=4,
        allow_writing_files=False,
        verbose=False,
    ).fit(raw, y, cat_features=cat_cols)
    test = pd.read_csv(ROOT / "data/test.csv")
    test_raw = test.drop(columns=["player_id"], errors="ignore").copy()
    test_raw[cat_cols] = test_raw[cat_cols].fillna("__MISSING__").astype(str)
    p_balanced_test = final_balanced.predict_proba(test_raw)

    full_cat = CatBoostClassifier()
    full_cat.load_model(str(ROOT / "models/segment-catboost.cbm"))
    p_cat = full_cat.predict_proba(test_raw)

    def engineered_features(frame):
        x = frame.drop(
            columns=[
                "segment",
                "player_id",
                "random_metric_1",
                "random_metric_2",
                "random_metric_3",
            ],
            errors="ignore",
        ).copy()
        ratios = {
            "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
            "spend_per_account_day": ("total_spending_thb", "account_age_days"),
            "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
            "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
            "gifts_per_friend": ("gifts_sent_received", "friend_count"),
            "invites_per_friend": ("friend_invites_sent", "friend_count"),
            "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
        }
        for name, (a, b) in ratios.items():
            x[name] = x[a] / (x[b].abs() + 1)
        x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
        x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
        x["missing_count"] = x.isna().sum(axis=1)
        return x.replace([np.inf, -np.inf], np.nan)

    bundle = joblib.load(ROOT / "models/segment-lightgbm.joblib")
    lightgbm_test = engineered_features(test)
    for col, dtype in bundle["categorical_dtypes"].items():
        lightgbm_test[col] = lightgbm_test[col].astype(dtype)
    p_lgb = bundle["model"].predict_proba(lightgbm_test)
    p_existing = (p_cat + p_lgb) / 2
    p_final = 0.8 * p_existing + 0.2 * p_balanced_test
    output = pd.DataFrame({"player_id": test.player_id, "segment": p_final.argmax(axis=1)})
    assert len(output) == 25889 and output.segment.isin(range(4)).all()
    final_balanced.save_model(str(ROOT / "models/segment-balanced-catboost.cbm"))
    output.to_csv(ROOT / "predictions/segment-balanced-catboost-test.csv", index=False)
    display(output.head())
    print(f"Saved {len(output):,} prediction rows. submit/index.csv unchanged.")
else:
    print("No test prediction generated: OOF score did not beat the current candidate.")

No test prediction generated: OOF score did not beat the current candidate.


## Research decision

Use the measured OOF comparison above to decide whether class weighting or its fixed low-weight blend is worth retaining. The previous XGBoost diversity and prior-correction screens did not improve the current candidate, so neither should enter the next-best ensemble.